In [29]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision.datasets import MNIST
from torchvision import transforms
from torch.utils.data import DataLoader


In [30]:
device = "cuda" if torch.cuda.is_available() else "cpu"

In [31]:

transform = transforms.ToTensor()
train_dataset = MNIST(
    root="./data",
    train=True,
    download=True,
    transform=transform
)
train_loader = DataLoader(
    train_dataset,
    batch_size=256,
    shuffle=True
)

In [32]:

# class LayerNet(nn.Module):
#     def __init__(self, in_dim, out_dim):
#         super().__init__()
#         self.layer = nn.Sequential(
#             nn.Linear(in_dim, out_dim),
#             nn.ReLU()
#         )

#     def forward(self, x):

#         h = self.layer(x)
#         h = F.normalize(h, dim=1)  # 防止退化

#         return h


# # 三层网络
# layers = [
#     LayerNet(784, 256).to(device),
#     LayerNet(256, 128).to(device),
#     LayerNet(128, 64).to(device)
# ]


model_1 = nn.Sequential(
    nn.Flatten(),
    nn.Linear(28*28, 2000),
    nn.ReLU(),
    nn.Linear(2000, 2000),
    nn.ReLU(),
    nn.Linear(2000, 2000),
    nn.ReLU(),
    nn.Linear(2000, 2000),
    nn.ReLU(),
    nn.Linear(2000, 10)
)

In [33]:
def class_matrix_loss(h, y, num_classes=10, lambda_inter=1):
    """Compute the class matrix loss.
    Args:
        h: [B, d] hidden representations
        y: [B] labels
        num_classes: number of classes
        lambda_inter: weight for inter-class loss"""


    B, d = h.shape
    # one-hot
    y_onehot = F.one_hot(y, num_classes).float()
    # 每类样本数
    count = y_onehot.sum(dim=0) + 1e-6

    # prototypes
    prototypes = (y_onehot.T @ h) / count.unsqueeze(1)

    # =================
    # 类内离散度
    # =================

    # 每个样本对应的聚类中心
    proto_expand = y_onehot @ prototypes

    intra_loss = ((h - proto_expand) ** 2).sum()/num_classes

    # =================
    # 类间 kernel
    # =================

    kernel_matrix = prototypes @ prototypes.T

    mask = 1 - torch.eye(num_classes, device=h.device)

    inter_loss = (kernel_matrix * mask).sum()/ (num_classes * (num_classes - 1))

    loss = intra_loss + lambda_inter * inter_loss

    return loss

In [34]:
def train(model, train_loader, device):

    epochs_per_layer = 10

    for layer_id in range(len(model)):

        if not isinstance(model[layer_id], nn.Linear):
            continue

        print("Training Layer", layer_id)

        optimizer = torch.optim.Adam(model[layer_id].parameters(), lr=1e-3)

        for epoch in range(epochs_per_layer):

            for x, y in train_loader:

                x = x.to(device)
                y = y.to(device)

                # forward到当前层
                with torch.no_grad():
                    for i in range(layer_id):
                        x = model[i](x)

                h = model[layer_id](x)

                loss = class_matrix_loss(h, y)

                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

            print("epoch:", epoch, "loss:", loss.item())

    print("Training Finished")

In [35]:
def predict(model, x):

    with torch.no_grad():
        h = model(x)
        pred = h.argmax(dim=1)
    return pred

In [36]:
def evaluate_accuracy(model, data_loader, device):
    model.eval()
    correct = 0
    total = 0

    with torch.no_grad():
        for x, y in data_loader:
            x = x.to(device)
            y = y.to(device)

            pred = predict(model, x)          # 你已有的函数
            correct += (pred == y).sum().item()
            total += y.size(0)

    acc = correct / total
    print(f"Accuracy: {acc:.4f} ({acc*100:.2f}%)")
    return acc

In [37]:
train(model_1.to(device), train_loader, device)

Training Layer 1
epoch: 0 loss: 1.330966830253601
epoch: 1 loss: 0.1642303168773651
epoch: 2 loss: 0.026578303426504135
epoch: 3 loss: 0.00505040492862463
epoch: 4 loss: 0.0010555831249803305
epoch: 5 loss: 0.0006599501939490438
epoch: 6 loss: 0.0005213054246269166
epoch: 7 loss: 0.0005196430720388889
epoch: 8 loss: 0.0007639303803443909
epoch: 9 loss: 0.0024779653176665306
Training Layer 3
epoch: 0 loss: 7.635240035597235e-05
epoch: 1 loss: 0.00010378292063251138
epoch: 2 loss: 0.00036134777474217117
epoch: 3 loss: 2.967788532259874e-05
epoch: 4 loss: 0.000335731019731611
epoch: 5 loss: 0.00023792742285877466
epoch: 6 loss: 0.0002145915204891935
epoch: 7 loss: 0.00013965583639219403
epoch: 8 loss: 3.9896476664580405e-05
epoch: 9 loss: 0.00012303533731028438
Training Layer 5
epoch: 0 loss: 0.00035537773510441184
epoch: 1 loss: 0.0007525546825490892
epoch: 2 loss: 0.0002428160805720836
epoch: 3 loss: 0.00011766807438107207
epoch: 4 loss: 0.0002220386959379539
epoch: 5 loss: 0.0001800044

In [38]:
evaluate_accuracy(model_1.to(device), train_loader, device)

Accuracy: 0.0878 (8.78%)


0.08778333333333334